In [0]:
%sql
CREATE OR REPLACE TABLE medical_insurance.gold.claims_analytics AS
WITH claim_base AS (
  SELECT 
    c.claim_id,
    c.patient_id,
    c.visit_id,
    c.hospital_id,
    c.claim_date,
    c.claim_amount,
    c.approved_amount,
    c.claim_status,
    c.claim_amount - c.approved_amount AS denied_amount
  FROM medical_insurance.silver.claim_silver c
),
claim_items_agg AS (
  SELECT 
    ci.claim_id,
    COUNT(DISTINCT ci.claim_item_id) AS total_items,
    COUNT(DISTINCT ci.procedure_code) AS procedure_count,
    COUNT(DISTINCT ci.drug_id) AS drug_count,
    SUM(ci.item_amount) AS total_item_amount,
    SUM(ci.item_quantity) AS total_quantity
  FROM medical_insurance.silver.claim_items_silver ci
  GROUP BY ci.claim_id
),
approval_info AS (
  SELECT 
    ca.claim_id,
    ca.reviewed_by,
    ca.approval_status,
    ca.approval_date,
    ca.rejection_reason,
    DATEDIFF(ca.approval_date, cb.claim_date) AS approval_turnaround_days
  FROM medical_insurance.silver.claim_approval_silver ca
  JOIN claim_base cb ON ca.claim_id = cb.claim_id
),
visit_info AS (
  SELECT 
    v.visit_id,
    v.visit_type,
    v.diagnosis_code,
    v.symptoms
  FROM medical_insurance.silver.visit_silver v
),
hospital_info AS (
  SELECT 
    h.hospital_id,
    h.hospital_name,
    h.hospital_type,
    h.governorate
  FROM medical_insurance.silver.hospital_silver h
)
SELECT 
  cb.claim_id,
  cb.patient_id,
  cb.visit_id,
  cb.hospital_id,
  h.hospital_name,
  h.hospital_type,
  h.governorate,
  cb.claim_date,
  YEAR(cb.claim_date) AS claim_year,
  MONTH(cb.claim_date) AS claim_month,
  QUARTER(cb.claim_date) AS claim_quarter,
  
  -- Visit context
  v.visit_type,
  v.diagnosis_code,
  v.symptoms,
  
  -- Claim amounts
  ROUND(cb.claim_amount, 2) AS claim_amount,
  ROUND(cb.approved_amount, 2) AS approved_amount,
  ROUND(cb.denied_amount, 2) AS denied_amount,
  cb.claim_status,
  
  -- Approval metrics
  CASE 
    WHEN cb.claim_amount > 0 THEN ROUND((cb.approved_amount * 100.0 / cb.claim_amount), 2)
    ELSE 0
  END AS approval_rate_pct,
  CASE 
    WHEN cb.claim_amount > 0 THEN ROUND((cb.denied_amount * 100.0 / cb.claim_amount), 2)
    ELSE 0
  END AS denial_rate_pct,
  
  -- Approval process details
  ai.reviewed_by,
  ai.approval_status,
  ai.approval_date,
  ai.rejection_reason,
  COALESCE(ai.approval_turnaround_days, 0) AS approval_turnaround_days,
  
  -- Claim items details
  COALESCE(cia.total_items, 0) AS total_claim_items,
  COALESCE(cia.procedure_count, 0) AS procedures_in_claim,
  COALESCE(cia.drug_count, 0) AS drugs_in_claim,
  ROUND(COALESCE(cia.total_item_amount, 0), 2) AS total_items_amount,
  COALESCE(cia.total_quantity, 0) AS total_items_quantity,
  
  -- Categorizations
  CASE 
    WHEN cb.claim_amount > 10000 THEN 'High Value'
    WHEN cb.claim_amount BETWEEN 5000 AND 10000 THEN 'Medium Value'
    WHEN cb.claim_amount < 5000 THEN 'Low Value'
    ELSE 'Unknown'
  END AS claim_value_category,
  
  CASE 
    WHEN cb.claim_status = 'Approved' AND cb.approved_amount = cb.claim_amount THEN 'Fully Approved'
    WHEN cb.claim_status = 'Approved' AND cb.approved_amount < cb.claim_amount THEN 'Partially Approved'
    WHEN cb.claim_status = 'Rejected' THEN 'Rejected'
    ELSE 'Unknown'
  END AS approval_category,
  
  CASE 
    WHEN ai.approval_turnaround_days <= 3 THEN 'Fast Processing'
    WHEN ai.approval_turnaround_days BETWEEN 4 AND 7 THEN 'Normal Processing'
    WHEN ai.approval_turnaround_days > 7 THEN 'Slow Processing'
    ELSE 'Unknown'
  END AS processing_speed_category,
  
  CURRENT_TIMESTAMP() AS created_at
  
FROM claim_base cb
LEFT JOIN claim_items_agg cia ON cb.claim_id = cia.claim_id
LEFT JOIN approval_info ai ON cb.claim_id = ai.claim_id
LEFT JOIN visit_info v ON cb.visit_id = v.visit_id
LEFT JOIN hospital_info h ON cb.hospital_id = h.hospital_id

In [0]:
%sql
-- Display sample records from claims analytics gold table
SELECT 
  claim_id,
  hospital_name,
  claim_date,
  visit_type,
  claim_amount,
  approved_amount,
  claim_status,
  approval_rate_pct,
  approval_turnaround_days,
  claim_value_category
FROM medical_insurance.gold.claims_analytics
ORDER BY claim_date DESC
LIMIT 10

In [0]:
%sql
-- Summary statistics by claim status and value category
SELECT 
  claim_status,
  claim_value_category,
  COUNT(DISTINCT claim_id) AS total_claims,
  ROUND(AVG(claim_amount), 2) AS avg_claim_amount,
  ROUND(AVG(approved_amount), 2) AS avg_approved_amount,
  ROUND(AVG(approval_rate_pct), 2) AS avg_approval_rate,
  ROUND(AVG(approval_turnaround_days), 2) AS avg_turnaround_days,
  ROUND(SUM(claim_amount), 2) AS total_claim_amount,
  ROUND(SUM(approved_amount), 2) AS total_approved_amount
FROM medical_insurance.gold.claims_analytics
GROUP BY claim_status, claim_value_category
ORDER BY claim_status, claim_value_category